# Notebook 03 — SmallCNN on CIFAR-10

**Note:** Training the seed-0 model takes ~20–40 min on CPU, ~5 min on GPU. The checkpoint is reloaded on subsequent runs.

## Structure
0. **Imports & setup**
1. **Configuration**
2. **Backward Factor Trace** — seed-0 model, BFT, exploratory plots (factor panels, conv kernels, spatial maps, scaffold, top stimuli)
3. **BFT figures** — main paper and appendix (placeholders)
4. **Fingerprints** — NNLS round-trip, ID sanity check, near-OOD (CIFAR-100), far-OOD (4 synthetic types), embeddings
5. **Fingerprint figures** — main paper and appendix (placeholders)

Validation, robustness and ablation analyses live in notebook 09.

## §0 — Imports & setup

In [ ]:
#%matplotlib inline
import sys, os, copy
sys.path.insert(0, '..')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from sklearn.metrics.pairwise import paired_cosine_distances
from torch.utils.data import DataLoader, TensorDataset

from src import (
    SmallCNN, load_experiment, save_experiment, collect_layer_dicts, bft,
    build_scaffold_edges, scaffold_loading_from_edges, scaffold_layer_sizes_from_edges,
    plot_scaffold_graph, extract_tree_nodes, plot_factor_tree, extract_fingerprint_matrix,
    compute_stimulus_similarity, project_stimuli_onto_tree, project_onto_bft,
    extract_factor_tree_nodes, compute_factor_activations, nodes_at_layer,
    plot_factor_overview_panel, plot_factor_gallery, plot_input_layer_factors,
    plot_embedding_comparison, plot_spatial_activation_maps, plot_similarity_heatmap,
    imdenorm as _imdenorm,
)

plt.rcParams.update({'figure.dpi': 80})
DEVICE = ('cuda' if torch.cuda.is_available() else
          'mps'  if torch.backends.mps.is_available() else 'cpu')
print('device:', DEVICE)

## §1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
MODEL_ROOT = '../data/models'
FIG_DIR    = '../figs/03_cnn_cifar10'
DATA_DIR   = '../data'
MODEL_DIR  = os.path.join(MODEL_ROOT, 'cifar10_cnn_seed0')
os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(FIG_DIR,    exist_ok=True)

# ── CIFAR-10 constants ────────────────────────────────────────────────────────
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                    'dog','frog','horse','ship','truck']
CLASS_NAMES     = {i: CIFAR10_CLASSES[i] for i in range(10)}
CIFAR10_MEAN    = (0.4914, 0.4822, 0.4465)
CIFAR10_STD     = (0.2470, 0.2435, 0.2616)
N_CLASSES       = 10
IMAGE_SIDE      = 32

# ── Training ──────────────────────────────────────────────────────────────────
N_EPOCHS = 150

# ── BFT hyperparameters ───────────────────────────────────────────────────────
# Selected by the nb09 S10 sweep, cluster run 4 (complete). See PUBLICATION_SETTINGS.md.
# K_MAX is FU1's "K@R²0.90" profile — the smallest per-layer rank reaching arbor-R² 0.90,
# with layers that never reach it keeping their auto-selected default. Against the old
# [6,6,6,6,10]: silhouette 0.329 → 0.356, fingerprint 188 → 159 dims, identical causal
# reconstruction (median preact-R² 0.938 on the classifier node), stability 0.862.
# NOTE the "rank ×0.7" rule that won on both MLPs and the ViT LOSES badly here
# (silhouette 0.247, R² 0.803, stability 0.792) — low rank does not transfer to conv arbors.
K_MAX           = [6, 7, 7, 10, 14]   # C0 circuit tree (nb15)
N_BRANCHES      = [1, 1, 1, 3, 14]   # C0 circuit tree (nb15)
POOL_METHOD     = 'avg'                 # inherited from nb03, never swept
STIM_THRESHOLD  = 0.0
N_TOP_PER_CLASS = 200   # C0 scale-up (was 60)

print('Config ready')
print(f'MODEL_DIR: {MODEL_DIR}')

# ── Fingerprint-tree HPs (paper/submission settings) — §4/§5 only ─────────────
K_MAX_FP     = [6, 6, 6, 6, 10]
N_BRANCHES_FP = [1, 1, 1, 1, 10]


## §2 — Backward Factor Trace (seed 0)

### 2a — Data, model and layer activations

In [ ]:
# ── Data and seed-0 model ─────────────────────────────────────────────────────
normalize    = T.Normalize(CIFAR10_MEAN, CIFAR10_STD)
train_tf     = T.Compose([T.RandomCrop(32, 4), T.RandomHorizontalFlip(), T.ToTensor(), normalize])
test_tf      = T.Compose([T.ToTensor(), normalize])
train_ds_aug = torchvision.datasets.CIFAR10(DATA_DIR, True,  download=True, transform=train_tf)
test_ds      = torchvision.datasets.CIFAR10(DATA_DIR, False, download=True, transform=test_tf)
train_loader = DataLoader(train_ds_aug, 128, shuffle=True,  num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,      256, shuffle=False, num_workers=0)

BASE_CONFIG = {
    'arch': 'SmallCNN',
    'arch_kwargs': {'channels': [32, 64, 128, 256], 'n_classes': 10, 'global_pool': True},
    'dataset': 'CIFAR10',
    'dataset_kwargs': {'root': '../data/', 'batch_size': 128},
    'label_transform': 'identity',
}


def _test_acc(m):
    m.eval()
    with torch.no_grad():
        n_correct = sum((m(x.to(DEVICE)).argmax(1) == y.to(DEVICE)).sum().item()
                        for x, y in test_loader)
    return n_correct / len(test_ds)


if os.path.exists(os.path.join(MODEL_DIR, 'weights.pt')):
    model0, cfg0 = load_experiment(MODEL_DIR, device=DEVICE)
    print(f'Loaded model from {MODEL_DIR} — test acc = {_test_acc(model0):.4f}')
else:
    print('No checkpoint found, training seed 0 (~20-40 min on CPU) …')
    torch.manual_seed(0)
    model0 = SmallCNN(channels=[32, 64, 128, 256], n_classes=10, global_pool=True).to(DEVICE)
    opt  = torch.optim.SGD(model0.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=N_EPOCHS)
    crit = nn.CrossEntropyLoss()
    for ep in range(1, N_EPOCHS + 1):
        model0.train()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = crit(model0(x), y); loss.backward(); opt.step()
        sch.step()
        if ep % 30 == 0 or ep == N_EPOCHS:
            print(f'  ep {ep}: test acc = {_test_acc(model0):.4f}')
    cfg0 = dict(BASE_CONFIG, description='SmallCNN on CIFAR-10, seed 0')
    save_experiment(model0, cfg0, MODEL_DIR)
    print(f'Saved to {MODEL_DIR}')

model0.eval()

In [ ]:
imdenorm = lambda img: _imdenorm(img, CIFAR10_MEAN, CIFAR10_STD)

def confidence_filter(raw, top_k, n_classes=N_CLASSES):
    keep = np.sort(np.concatenate([
        np.where(raw['targets'] == c)[0][
            np.argsort(raw['confidences'][raw['targets'] == c])[::-1][:top_k]]
        for c in range(n_classes)]))
    images = raw['images'][keep]
    targets = raw['targets'][keep]
    layer_data = [{**ld, 'input_fmap': ld['input_fmap'][keep],
                   'output_fmap': ld['output_fmap'][keep]}
                   for ld in raw['layer_data']]
    return {'images': images, 'targets': targets,
            'confidences': raw['confidences'][keep],
            'layer_data': layer_data}, keep

print('Collecting layer data for seed 0 …')
raw0 = collect_layer_dicts(model0, test_loader, DEVICE, only_correct=True)

data0, _ = confidence_filter(raw0, N_TOP_PER_CLASS)
all_images0  = data0['images']
all_targets0 = data0['targets']
layer_inputs0 = [ld['input_fmap'] for ld in data0['layer_data']]
n_samples0   = len(all_images0)
print(f'{n_samples0} samples after filter | layers: {[x.shape for x in layer_inputs0]}')

### 2b — Run BFT

In [ ]:
# ── BFT (seed 0, layer-dict mode) ─────────────────────────────────────────────

from src import cached_tree
tree_root0 = cached_tree('nb03_circuit', lambda: bft(
    data0['layer_data'],
    k_max=K_MAX, n_branches=N_BRANCHES,
    conv_pool_method=POOL_METHOD,
    stimulus_threshold=STIM_THRESHOLD,
    weighting='img_selectivity', verbose=1, n_jobs=3,
), params=dict(k=K_MAX, b=N_BRANCHES, tau=STIM_THRESHOLD, n=len(all_targets0)))

tree_nodes0   = extract_tree_nodes(tree_root0)
factor_nodes0 = extract_factor_tree_nodes(tree_root0)
l0_nodes0     = nodes_at_layer(tree_root0, 0)
print(f'Tree nodes: {len(tree_nodes0)}  factor nodes: {len(factor_nodes0)}')
print(f'Root factors: K={len(tree_root0.root.lambdas)}')
print(f'L0 leaf nodes (first conv): {len(l0_nodes0)}')


### 2c — Exploratory plots: factor overview panels and galleries

In [ ]:
# ── Plot 1+4: Factor overview panels & per-factor galleries (all tree nodes) ──
for node in tree_nodes0:
    layer_name = node.layer_name if node.layer_name else f'L{node.layer_idx}'
    path_label = 'F' + '-F'.join(str(f) for f in node.path) if node.path else 'root'
    node_id = f"{layer_name}_{path_label}"
    figs1 = plot_factor_overview_panel(node, all_images0, all_targets0, CLASS_NAMES)
    for k, fig in enumerate(figs1):
        fig.savefig(os.path.join(FIG_DIR, f'factor_overview_{node_id}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)
    K = node.img_factors.shape[1]
    for k in range(K):
        fig4 = plot_factor_gallery(node, all_images0, all_targets0, CLASS_NAMES, k=k, n=10)
        fig4.savefig(os.path.join(FIG_DIR, f'factor_gallery_{node_id}_k{k}.pdf'),
                     bbox_inches='tight')
        plt.close(fig4)

print(f'Plot 1+4 saved for {len(tree_nodes0)} tree nodes.')

### 2d — Exploratory plots: input-layer spatial factors

In [ ]:
# ── Plot 2: Input-layer spatial factors (conv_rgb for first-conv leaf nodes) ──
for leaf in l0_nodes0:
    layer_name = leaf.layer_name if leaf.layer_name else f'L{leaf.layer_idx}'
    path_label = 'F' + '-F'.join(str(f) for f in leaf.path) if leaf.path else 'root'
    node_id = f"{layer_name}_{path_label}"
    figs2 = plot_input_layer_factors(leaf, all_images0, arch='conv_rgb',
                                      image_shape=(3, IMAGE_SIDE, IMAGE_SIDE))
    for k, fig in enumerate(figs2):
        fig.savefig(os.path.join(FIG_DIR, f'input_factors_{node_id}_k{k}.pdf'),
                    bbox_inches='tight')
        plt.close(fig)

print(f'Plot 2 saved for {len(l0_nodes0)} L0 leaf nodes.')

### 2e — Exploratory plots: spatial activation maps

In [ ]:
# ── Spatial activation maps (top conv layer in spine) ─────────────────────────
for leaf in l0_nodes0[:2]:
    if leaf.layer_type != 'conv':
        continue
    path_label = 'F' + '-F'.join(str(f) for f in leaf.path) if leaf.path else 'root'
    node_id = f"{leaf.layer_name}_{path_label}"
    fig = plot_spatial_activation_maps(model0, all_images0, leaf, data0['layer_data'], DEVICE,
                                        n_images=6, denorm_fn=imdenorm,
                                        title=f'{node_id}: spatial activation maps')
    plt.savefig(os.path.join(FIG_DIR, f'spatial_{node_id}.pdf'), bbox_inches='tight')
    plt.show()


### 2f — Exploratory plots: scaffold graph

In [ ]:
# ── Scaffold graph (seed 0) ───────────────────────────────────────────────────
def _get_spine(root):
    chain, node = [root], root
    while node.children:
        node = node.children[0]; chain.append(node)
    return chain

spine = _get_spine(tree_root0.root)
layer_results = list(reversed(spine))          # forward order: input → output
edge_matrices, neg_edge_matrices = build_scaffold_edges(
    layer_results[1:], fi="path", fi_seed=layer_results[0], top_pct=0.05,
)
scaffold_loading = scaffold_loading_from_edges(edge_matrices)
layer_sizes = scaffold_layer_sizes_from_edges(edge_matrices)
fig = plot_scaffold_graph(scaffold_loading, edge_matrices, layer_sizes,
                          neg_edge_matrices=neg_edge_matrices)
fig.axes[0].set_title("Scaffold graph — seed 0 (node colour = dominant CIFAR-10 class)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "scaffold_seed0.pdf"), bbox_inches="tight")
plt.show()


### 2g — Exploratory plots: top stimuli per output factor

In [ ]:
# ── Top-N stimulus gallery per output factor ──────────────────────────────────
K_root = tree_root0.root.img_factors.shape[1]
N_GAL = 8
fig, axes = plt.subplots(K_root, N_GAL, figsize=(N_GAL * 1.6, K_root * 1.8))
for k in range(K_root):
    top_idx = np.argsort(tree_root0.root.img_factors[:, k])[::-1][:N_GAL]
    for col, idx in enumerate(top_idx):
        axes[k, col].imshow(imdenorm(all_images0[idx]))
        axes[k, col].set_title(CIFAR10_CLASSES[all_targets0[idx]][:4], fontsize=7)
        axes[k, col].axis("off")
    axes[k, 0].set_ylabel(f"k={k} λ={tree_root0.root.lambdas[k]:.2f}", fontsize=8,
                           rotation=0, labelpad=50, va="center")
plt.suptitle("Top stimuli per output factor (seed 0)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "top_stimuli_gallery.pdf"), bbox_inches="tight")
plt.show()


## §3 — BFT figures (main paper & appendix)

### 3a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the CIFAR-10 CNN circuits
# Requires: tree_root0, model0, all_images0, all_targets0  (§2 above)
# Writes:   figures/figdata/nb03_circuits.npz  (+ .json)
# The figures are built in notebooks/fig03_cnn_cifar.ipynb from this bundle alone,
# so export a superset: every factor, both signs, class profiles, example stimuli.
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport

CLASSES = list(range(N_CLASSES))
root = tree_root0.root


def spatial_maps(node, n_images=6):
    """Top/bottom stimuli of factor 0 and their channel-weighted activation maps.

    Needs the model, so it can only be computed here — a conv figure that shows
    *where* a circuit fires would otherwise be impossible off-cluster.
    """
    C_out = node.weight.shape[0]
    ch = np.maximum(node.connection_factors[:, 0].reshape(C_out, -1).sum(1), 0)
    ch = ch / (ch.sum() + 1e-12)
    scores = node.img_factors[:, 0]
    sel = dict(top=np.argsort(scores)[::-1][:n_images], bottom=np.argsort(scores)[:n_images])
    out = {}
    for tag, idx in sel.items():
        store = {}
        mod = dict(model0.named_modules())[node.layer_name]
        h = mod.register_forward_hook(lambda m, i, o: store.update(out=o.detach().cpu()))
        with torch.no_grad():
            model0(torch.from_numpy(all_images0[idx]).float().to(DEVICE))
        h.remove()
        fm = store['out'].numpy()
        out[tag] = dict(index=idx, images=all_images0[idx].astype(np.float32),
                        maps=np.maximum((fm * ch[None, :, None, None]).sum(1), 0))
    return out


# ── one circuit per output-layer factor, traced to the input layer ──────────
circuits = []
for child in root.children:
    chain, leaf = [root, child], child
    while leaf.children:
        leaf = leaf.children[0]
        chain.append(leaf)
    fwd = list(reversed(chain))                                   # input-first
    E, negE = build_scaffold_edges(fwd[1:], fi='path', fi_seed=fwd[0], top_pct=0.05)
    prof = figexport.class_profile(root, all_targets0, CLASSES)[int(child.path[0])]
    circuits.append(dict(
        k=int(child.path[0]), profile=prof,
        top_classes=[int(c) for c in np.argsort(prof)[::-1] if prof[c] >= 0.15],
        leaf_layer=str(leaf.layer_name), depth=len(chain),
        scaffold=figexport.scaffold_summary(
            E, negE, scaffold_loading_from_edges(E),
            scaffold_layer_sizes_from_edges(E))))

# ── the whole trace, plus what only the model can give us ──────────────────
_stim = figexport.subsample_by_class(all_targets0, CLASSES, 100, seed=0)
nodes = figexport.export_tree(root, labels=all_targets0, classes=CLASSES,
                              images=all_images0, stim_idx=_stim,
                              img_max_side=32, example_max_side=32)
_bfs, _q = [], [root]
while _q:
    _n = _q.pop(0); _bfs.append(_n); _q.extend(_n.children)
for _d, _n in zip(nodes, _bfs):
    if _n.layer_type == 'conv':
        try:
            _d['spatial'] = spatial_maps(_n)
        except Exception as _e:                     # never lose the bundle over a hook
            print(f'  spatial maps skipped for {_n.layer_name}: {_e}')

D = figdata.save('nb03_circuits', dict(
    n_classes=N_CLASSES, class_names=[str(c) for c in CIFAR10_CLASSES],
    image_mean=list(CIFAR10_MEAN), image_std=list(CIFAR10_STD),
    meta=figexport.trace_meta(tree_root0, k_max=K_MAX, n_branches=N_BRANCHES,
                              pool_method=POOL_METHOD,
                              stimulus_threshold=STIM_THRESHOLD),
    circuits=circuits, nodes=nodes,
    stimuli=figexport.example_stimuli(all_images0, all_targets0, CLASSES,
                                      per_class=8, max_side=32)))
figdata.summary('nb03_circuits')


### 3b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig03_cnn_cifar.ipynb from the exported bundle)


## §4 — Fingerprints

In [ ]:
# ── Two trees: circuits use the C0 tree (above); fingerprints use a shallower
#    paper-HP tree (nb16 C1.8). Both cached.
from src import cached_tree
tree_circuit = tree_root0                    # keep the C0 circuit tree for §7/§8
tree_root0 = cached_tree('nb03_fp', lambda: bft(
    data0['layer_data'], k_max=K_MAX_FP, n_branches=N_BRANCHES_FP,
    conv_pool_method=POOL_METHOD, stimulus_threshold=STIM_THRESHOLD, weighting='img_selectivity', n_jobs=3),
    params=dict(k=K_MAX_FP, b=N_BRANCHES_FP, tau=STIM_THRESHOLD, n=len(all_targets0)))
print('circuit tree:', sum(1 for _ in tree_circuit.nodes()), 'nodes | '
      'fingerprint tree:', sum(1 for _ in tree_root0.nodes()), 'nodes')

ANALYSIS_CTX = dict(
    tag='nb03', model=model0, tree_circuit=tree_circuit, tree_fp=tree_root0,
    layer_inputs=[d['input_fmap'] for d in data0['layer_data']], labels_task=all_targets0.astype(int),
    labels_fine=all_targets0.astype(int), eval_loader=test_loader,
    label_transform=None, device=DEVICE,
    layer_names=[d['name'] for d in data0['layer_data']],
    n_classes=10, k_cap=16, last_extra=4, k_max_cfg=K_MAX,
    prune_fractions=(0.02, 0.05, 0.1, 0.2), n_random=5, stab_seeds=5)

In [ ]:
# §3 only exports plot data — the paper figures live in
# notebooks/fig03_cnn_cifar.ipynb — so nothing above changed matplotlib's rcParams.
# Kept so the exploratory plots below render at screen size.
plt.rcParams.update({'figure.dpi': 80})


### 4a — NNLS round-trip fidelity

In [ ]:
# ── Round-trip test ───────────────────────────────────────────────────────────
N_RT = min(200, n_samples0)
rng_rt = np.random.default_rng(0)
rt_sub = rng_rt.choice(n_samples0, N_RT, replace=False)
rt_inputs = [l[rt_sub] for l in layer_inputs0]

projected_rt = project_stimuli_onto_tree(tree_root0, rt_inputs)
F_orig_rt    = extract_fingerprint_matrix(tree_root0, rt_sub)
F_rt         = extract_fingerprint_matrix(projected_rt, np.arange(N_RT))
rt_sims      = 1.0 - paired_cosine_distances(F_orig_rt, F_rt)
rt_root      = 1.0 - paired_cosine_distances(
    tree_root0.root.img_factors[rt_sub], projected_rt.img_factors)
print(f'Round-trip (full):  mean={rt_sims.mean():.4f}  std={rt_sims.std():.4f}')
print(f'Round-trip (root):  mean={rt_root.mean():.4f}  std={rt_root.std():.4f}')
print()
print('Active-sample fractions per node:')
for tn in tree_nodes0[:8]:  # show first 8 nodes
    sw = tn['stimulus_weights']
    print(f'  layer={tn["layer_idx"]}  path={tn["path"]}  active={(sw > 0.01).mean():.3f}')


### 4b — ID sanity check: train vs test cross-similarity

In [ ]:
# ── ID sanity check (train set) ───────────────────────────────────────────────
train_ds_eval = torchvision.datasets.CIFAR10(DATA_DIR, True, download=True, transform=test_tf)
train_loader_eval = DataLoader(train_ds_eval, 256, shuffle=False, num_workers=0)

print('Collecting training set layer data …')
raw_train = collect_layer_dicts(model0, train_loader_eval, DEVICE, only_correct=True)
data_train, _ = confidence_filter(raw_train, N_TOP_PER_CLASS)
train_layer_inputs = [ld['input_fmap'] for ld in data_train['layer_data']]
train_targets = data_train['targets']
print(f'Training samples: {len(train_targets)}')

projected_train  = project_stimuli_onto_tree(tree_root0, train_layer_inputs)
factor_nodes_tr  = extract_factor_tree_nodes(projected_train)

# Cross-similarity matrix: 5 representative classes, train vs test
N_BLK = 50; SHOW_CL = [0, 1, 3, 7, 8]; rng_id = np.random.default_rng(42)
blocks = {}
for cl in SHOW_CL:
    tr_cl = np.where(train_targets == cl)[0]
    if len(tr_cl) > N_BLK: tr_cl = rng_id.choice(tr_cl, N_BLK, replace=False)
    blocks[f'tr-{CIFAR10_CLASSES[cl][:4]}'] = extract_fingerprint_matrix(projected_train, tr_cl)
    te_cl = np.where(all_targets0 == cl)[0]
    if len(te_cl) > N_BLK: te_cl = rng_id.choice(te_cl, N_BLK, replace=False)
    blocks[f'te-{CIFAR10_CLASSES[cl][:4]}'] = extract_fingerprint_matrix(tree_root0, te_cl)

F_cross = np.concatenate(list(blocks.values()), axis=0)
bl_sizes = [len(v) for v in blocks.values()]
bl_starts = [0] + list(np.cumsum(bl_sizes[:-1]))
bl_ends   = list(np.cumsum(bl_sizes))
S_cross   = compute_stimulus_similarity(F_cross)
centres   = np.array(bl_starts) + np.array(bl_sizes) / 2

fig = plot_similarity_heatmap(S_cross, bl_sizes, list(blocks),
                               title='ID Sanity Check: train vs test cross-similarity (5 classes)')
plt.savefig(os.path.join(FIG_DIR, 'id_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

### 4c — Near-OOD: CIFAR-100

In [ ]:
# ── Near-OOD: CIFAR-100 ────────────────────────────────────────────────────────
cifar100_test = torchvision.datasets.CIFAR100(
    DATA_DIR, train=False, download=True,
    transform=T.Compose([T.ToTensor(), normalize])  # apply CIFAR-10 normalisation
)
cifar100_loader = DataLoader(cifar100_test, 256, shuffle=False, num_workers=0)

print('Projecting CIFAR-100 onto BFT tree …')
projected_ood    = project_onto_bft(tree_root0, model0, cifar100_loader, only_correct=False, device=DEVICE)
ood_images       = projected_ood.images
ood_targets      = projected_ood.targets
n_ood            = len(ood_images)
factor_nodes_ood = extract_factor_tree_nodes(projected_ood)

# Collect raw layer activations for MDS comparison (cell 25)
raw_ood          = collect_layer_dicts(model0, cifar100_loader, DEVICE, only_correct=False)
ood_layer_inputs = [ld['input_fmap'] for ld in raw_ood['layer_data']]

# Model predictions (CIFAR-10 classes) for factor tree grouping
model0.eval()
ood_preds = np.concatenate([
    model0(x.to(DEVICE)).argmax(1).cpu().numpy()
    for x, _ in cifar100_loader
])
print(f'CIFAR-100 samples: {n_ood}')
print(f'Pred distribution: {dict(zip(*np.unique(ood_preds, return_counts=True)))}')

# Factor tree per predicted CIFAR-10 class
fig, axes = plt.subplots(2, 5, figsize=(30, 8), squeeze=False)
for d in range(N_CLASSES):
    ax = axes[d // 5][d % 5]
    cl_idx = np.where(ood_preds == d)[0]
    acts = compute_factor_activations(factor_nodes_ood, cl_idx)
    plot_factor_tree(factor_nodes_ood, acts, ax=ax,
                     title=f'OOD→pred {CIFAR10_CLASSES[d]}\nn={len(cl_idx)}')
plt.suptitle('CIFAR-100 OOD — factor tree per predicted CIFAR-10 class', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'near_ood_trees.pdf'), bbox_inches='tight')
plt.show()

### 4d — Fingerprint cross-similarity: CIFAR-10 vs CIFAR-100

In [ ]:
# ── Cross-similarity: ID CIFAR-10 vs CIFAR-100 ───────────────────────────────
N_BLOCK = 40; rng_blk = np.random.default_rng(2)
BLOCK_ID  = [0, 1, 7, 8]  # airplane, auto, horse, ship
BLOCK_C100 = [0, 10, 50, 90]  # a sample of CIFAR-100 classes

blocks_ood = {}
for cl in BLOCK_ID:
    idx = rng_blk.choice(np.where(all_targets0 == cl)[0],
                          min(N_BLOCK, (all_targets0 == cl).sum()), replace=False)
    blocks_ood[f'C10-{CIFAR10_CLASSES[cl][:4]}'] = extract_fingerprint_matrix(tree_root0, idx)
for c100 in BLOCK_C100:
    idx = np.where(ood_targets == c100)[0]
    if len(idx) == 0: continue
    idx = rng_blk.choice(idx, min(N_BLOCK, len(idx)), replace=False)
    blocks_ood[f'C100-{c100}'] = extract_fingerprint_matrix(projected_ood, idx)

F_blk  = np.concatenate(list(blocks_ood.values()), axis=0)
bl_s   = [len(v) for v in blocks_ood.values()]
bl_st  = [0] + list(np.cumsum(bl_s[:-1]))
bl_en  = list(np.cumsum(bl_s))
S_blk  = compute_stimulus_similarity(F_blk)
ctr    = np.array(bl_st) + np.array(bl_s) / 2

fig = plot_similarity_heatmap(S_blk, bl_s, list(blocks_ood),
                               title='ID CIFAR-10 vs OOD CIFAR-100 fingerprint cross-similarity')
plt.savefig(os.path.join(FIG_DIR, 'near_ood_cross_similarity.pdf'), bbox_inches='tight')
plt.show()

### 4e — Far-OOD: synthetic images

In [ ]:
# ── Far-OOD: 4 synthetic 3-channel image types ────────────────────────────────
IMAGE_SIDE = 32; C = 3
N_FAR = 200; rng_f = np.random.default_rng(99)
_mn = np.array(CIFAR10_MEAN)[:, None, None]
_st = np.array(CIFAR10_STD)[:, None, None]
_chk3 = (np.indices((IMAGE_SIDE, IMAGE_SIDE)).sum(axis=0) % 2)[None].astype(np.float32)

_orig     = all_images0[:N_FAR] * _st + _mn
_inverted = np.clip(1.0 - _orig, 0, 1)
_inv_norm = (_inverted - _mn) / _st

far_ood_arrays = {
    'gaussian_noise': np.clip(rng_f.normal(0.5, 0.25, (N_FAR, C, IMAGE_SIDE, IMAGE_SIDE)).astype(np.float32), 0, 1),
    'uniform_gray':   np.full((N_FAR, C, IMAGE_SIDE, IMAGE_SIDE), 0.5, dtype=np.float32),
    'checkerboard':   np.broadcast_to(_chk3, (N_FAR, C, IMAGE_SIDE, IMAGE_SIDE)).copy().astype(np.float32),
    'inverted_test':  _inv_norm.astype(np.float32),
}

far_ood_data = {}
for name, imgs in far_ood_arrays.items():
    ds     = TensorDataset(torch.from_numpy(imgs), torch.zeros(len(imgs), dtype=torch.long))
    loader = DataLoader(ds, 128, shuffle=False)
    raw    = collect_layer_dicts(model0, loader, DEVICE, only_correct=False)
    d = dict(raw)
    d['layer_inputs']   = [ld['input_fmap'] for ld in raw['layer_data']]
    d['projected_root'] = project_onto_bft(tree_root0, model0, loader, only_correct=False, device=DEVICE)
    d['factor_nodes']   = extract_factor_tree_nodes(d['projected_root'])
    far_ood_data[name]  = d
    print(f'{name:20s}  n={len(imgs)}')

# Example images
n_ex = 6
fig, axes = plt.subplots(len(far_ood_arrays), n_ex,
                          figsize=(n_ex * 2, len(far_ood_arrays) * 2.2))
for row, (name, imgs) in enumerate(far_ood_arrays.items()):
    for col in range(n_ex):
        if name == 'inverted_test':
            axes[row, col].imshow(imdenorm(imgs[col]))
        else:
            axes[row, col].imshow(np.clip(imgs[col].transpose(1, 2, 0), 0, 1))
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(name, fontsize=9, rotation=30, ha='right', va='center')
plt.suptitle('Far OOD — example images (3-channel RGB)', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_examples.pdf'), bbox_inches='tight')
plt.show()

# Factor tree per far-OOD type
n_types = len(far_ood_data)
fig, axes = plt.subplots(1, n_types, figsize=(6 * n_types, 4.5))
for ax, (name, d) in zip(axes, far_ood_data.items()):
    all_idx = np.arange(len(d['images']))
    acts    = compute_factor_activations(d['factor_nodes'], all_idx)
    plot_factor_tree(d['factor_nodes'], acts, ax=ax, title=name)
plt.suptitle('Far OOD — factor tree activations', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'far_ood_trees.pdf'), bbox_inches='tight')
plt.show()

### 4f — Fingerprint embeddings and intra/inter-class similarity

In [ ]:
# ── Plot 7: embedding comparison (PCA fingerprints | PCA last-layer | PCA all-layers | MDS fingerprints) ─
N_EACH = 60; rng_m = np.random.default_rng(7)

def _pool_fmap(a, n):
    return a.mean(axis=(2, 3)) if a.ndim == 4 else a.reshape(n, -1)

F_parts, la_parts, full_la_parts, emb_labels, emb_conditions = [], [], [], [], []

id_sub = rng_m.choice(n_samples0, min(N_EACH, n_samples0), replace=False)
F_parts.append(extract_fingerprint_matrix(tree_root0, id_sub))
la_parts.append(layer_inputs0[-1][id_sub].reshape(len(id_sub), -1))
full_la_parts.append(np.concatenate([_pool_fmap(layer_inputs0[i][id_sub], len(id_sub))
                                      for i in range(len(layer_inputs0))], axis=1))
emb_labels.extend(all_targets0[id_sub].tolist())
emb_conditions.extend(['ID-CIFAR10'] * len(id_sub))

ood_sub = rng_m.choice(n_ood, min(N_EACH, n_ood), replace=False)
F_parts.append(extract_fingerprint_matrix(projected_ood, ood_sub))
la_parts.append(ood_layer_inputs[-1][ood_sub].reshape(len(ood_sub), -1))
full_la_parts.append(np.concatenate([_pool_fmap(ood_layer_inputs[i][ood_sub], len(ood_sub))
                                      for i in range(len(ood_layer_inputs))], axis=1))
emb_labels.extend([N_CLASSES] * len(ood_sub))
emb_conditions.extend(['OOD-CIFAR100'] * len(ood_sub))

for name, d in far_ood_data.items():
    sub = rng_m.choice(len(d['images']), min(N_EACH, len(d['images'])), replace=False)
    F_parts.append(extract_fingerprint_matrix(d['projected_root'], sub))
    la_parts.append(d['layer_inputs'][-1][sub].reshape(len(sub), -1))
    full_la_parts.append(np.concatenate([_pool_fmap(d['layer_inputs'][i][sub], len(sub))
                                          for i in range(len(d['layer_inputs']))], axis=1))
    emb_labels.extend([N_CLASSES + 1] * len(sub))
    emb_conditions.extend([name] * len(sub))

F_joint = np.concatenate(F_parts, axis=0)
la_joint = np.vstack(la_parts)
emb_labels = np.array(emb_labels)

full_class_names = {i: CIFAR10_CLASSES[i] for i in range(N_CLASSES)}
full_class_names[N_CLASSES]     = 'CIFAR-100'
full_class_names[N_CLASSES + 1] = 'Far-OOD'

fig7 = plot_embedding_comparison(
    F_joint, la_joint, emb_labels, full_class_names,
    condition_labels=emb_conditions,
    far_ood_conditions=list(far_ood_data.keys()),
    activations_all=np.concatenate(full_la_parts, axis=0),
    title='ID CIFAR-10 / CIFAR-100 / Far-OOD')
fig7.savefig(os.path.join(FIG_DIR, 'embedding_comparison.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig7)

# Fingerprint intra vs inter-class similarity histogram
F_all = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
S_all = compute_stimulus_similarity(F_all)
intra_vals, inter_vals = [], []
for ci in range(N_CLASSES):
    mask = all_targets0 == ci
    intra = S_all[np.ix_(mask, mask)]
    intra_vals.extend(intra[np.triu_indices_from(intra, k=1)])
    for cj in range(ci + 1, N_CLASSES):
        inter_vals.extend(S_all[np.ix_(mask, all_targets0 == cj)].ravel())
intra_arr, inter_arr = np.array(intra_vals), np.array(inter_vals)
print(f'Intra-class: {intra_arr.mean():.3f} ± {intra_arr.std():.3f}')
print(f'Inter-class: {inter_arr.mean():.3f} ± {inter_arr.std():.3f}')
fig_ii, ax = plt.subplots(figsize=(6, 4))
ax.hist(intra_arr, bins=60, alpha=0.6, label='Intra-class', density=True)
ax.hist(inter_arr, bins=60, alpha=0.6, label='Inter-class', density=True)
ax.axvline(intra_arr.mean(), color='C0', ls='--')
ax.axvline(inter_arr.mean(), color='C1', ls='--')
ax.set(xlabel='Cosine similarity', ylabel='Density',
       title='Factor fingerprint: intra vs inter-class similarity')
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'fingerprint_intra_inter.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_ii)

In [ ]:
# ── Plot 7b: Same 4-panel embedding, ID test data only ──────────────────────
def _pool_fmap_id(a):
    return a.mean(axis=(2, 3)) if a.ndim == 4 else a.reshape(len(a), -1)

F_id   = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
act_id = layer_inputs0[-1].reshape(n_samples0, -1)
act_id_all = np.concatenate([_pool_fmap_id(layer_inputs0[i])
                              for i in range(len(layer_inputs0))], axis=1)

fig_id = plot_embedding_comparison(
    F_id, act_id, all_targets0, CLASS_NAMES,
    activations_all=act_id_all,
    title='ID test data — BFT fingerprint embeddings',
)
fig_id.savefig(os.path.join(FIG_DIR, 'embedding_id_only.pdf'), bbox_inches='tight')
plt.show(); plt.close(fig_id)

## §5 — Fingerprint figures (main paper & appendix)

### 5a — Main-paper figure

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EXPORT — figure data for the CIFAR-10 CNN fingerprints
# Requires: tree_root0, all_targets0, n_samples0  (§2), rt_sims (§4a),
#           projected_train, train_targets (§4b), projected_ood, ood_targets,
#           ood_preds, n_ood (§4c), far_ood_data (§4e)
# Writes:   figures/figdata/nb03_fingerprints.npz  (+ .json)
# ═══════════════════════════════════════════════════════════════════════════════
from src import figdata, figexport
from src.paper_figures import unit

CLASSES = list(range(N_CLASSES))
FAR_LABEL = {'gaussian_noise': 'noise', 'uniform_gray': 'gray',
             'checkerboard': 'checker', 'inverted_test': 'inverted'}

F       = extract_fingerprint_matrix(tree_root0, np.arange(n_samples0))
F_train = extract_fingerprint_matrix(projected_train, np.arange(len(train_targets)))
F_ood   = extract_fingerprint_matrix(projected_ood, np.arange(n_ood))
F_far   = {n: extract_fingerprint_matrix(d['projected_root'],
                                         np.arange(len(d['images'])))
           for n, d in far_ood_data.items()}

# which fingerprint dim belongs to which (layer, root branch, factor) — the
# column order extract_fingerprint_matrix produces (BFS over the tree)
_dims, _q = [], [tree_root0.root]
while _q:
    _n = _q.pop(0)
    for _k in range(_n.img_factors.shape[1]):
        _dims.append((_n.layer_idx, _n.path[0] if _n.path else -1, _k))
    _q.extend(_n.children)
_dims = np.array(_dims)
_top  = _dims[:, 0].max()
BLOCKS = [[int(r)] + [i for i in range(len(_dims))
                      if _dims[i, 0] != _top and _dims[i, 1] == _dims[r, 2]]
          for r in np.where(_dims[:, 0] == _top)[0]]


def centroid_cos(Fm):
    U = unit(Fm)
    c = U.mean(0)
    return U @ (c / (np.linalg.norm(c) + 1e-12))


CENT = unit(np.stack([F[all_targets0 == c].mean(0) for c in CLASSES]))
COND = ([dict(label='ID test', values=centroid_cos(F), color_key='id_data'),
         dict(label='CIFAR-100', values=centroid_cos(F_ood), color_key='near_ood')] +
        [dict(label=FAR_LABEL.get(n, n), values=centroid_cos(F_far[n]),
              color_key='far_ood') for n in far_ood_data])
LIKE = [dict(label=lab, values=(unit(X) @ CENT.T).max(1), color_key=ck)
        for lab, X, ck in ([('ID test', F, 'id_data'), ('CIFAR-100', F_ood, 'near_ood')] +
                           [(FAR_LABEL.get(n, n), F_far[n], 'far_ood')
                            for n in far_ood_data])]

_sub     = figexport.subsample_by_class(all_targets0, CLASSES, 150, seed=0)
_sub_tr  = figexport.subsample_by_class(train_targets, CLASSES, 150, seed=0)
_sub_ood = np.arange(min(1500, len(F_ood)))

D = figdata.save('nb03_fingerprints', dict(
    n_classes=N_CLASSES, class_names=[str(c) for c in CIFAR10_CLASSES],
    n_factors=F.shape[1], dims=_dims,
    col_order=[c for b in BLOCKS for c in b],
    blk_edge=np.cumsum([len(b) for b in BLOCKS])[:-1],
    block_sizes=[len(b) for b in BLOCKS],
    fp_mean_by_class=np.stack([F[all_targets0 == c].mean(0) for c in CLASSES]),
    fp_mean_ood=F_ood.mean(0),
    cond=COND, like=LIKE, rt_sims=rt_sims,
    fp=dict(id=F[_sub].astype(np.float32), id_targets=all_targets0[_sub],
            id_index=_sub,
            train=F_train[_sub_tr].astype(np.float32),
            train_targets=np.asarray(train_targets)[_sub_tr],
            ood=F_ood[_sub_ood].astype(np.float32),
            ood_targets=np.asarray(ood_targets)[_sub_ood],
            ood_preds=np.asarray(ood_preds)[_sub_ood],
            far=[dict(label=FAR_LABEL.get(n, n), F=F_far[n][:300].astype(np.float32))
                 for n in far_ood_data])))
figdata.summary('nb03_fingerprints')


### 5b — Appendix figure

In [ ]:
# (appendix figure: build it in notebooks/fig03_cnn_cifar.ipynb from the exported bundle)


## §6 — Hyperparameter check (held-out arbor R²)

Re-derives the per-layer circuit rank with the metric-free C0 rule (`src.hp_selection`), on the circuit tree's own arbors. Confirms the `K_MAX` in §1 sits at the reconstruction plateau; reads no fingerprint metric. Cached.

In [ ]:
from src import node_pos_arbor, nodes_per_layer, select_ranks, cached_result
_C = ANALYSIS_CTX
_npl = nodes_per_layer(_C['tree_circuit'], max_nodes=2)
_arbors = {li: [node_pos_arbor(nd, _C['layer_inputs'][li]) for nd in nds]
           for li, nds in _npl.items()}
hp_sel = cached_result(
    _C['tag'] + '_hpsel',
    lambda: select_ranks(_arbors, _C['labels_task'], k_cap=_C['k_cap'],
                         n_classes=_C['n_classes'], last_extra=_C['last_extra']),
    params=dict(kcap=_C['k_cap'], n=len(_C['labels_task']),
                kmax=list(_C['tree_circuit'].root.lambdas.shape)))
print('held-out K* per layer:', hp_sel['profile']['k_from_criterion'])
print('assembled profile     k_max=%s  n_branches=%s'
      % (hp_sel['profile']['k_max'], hp_sel['profile']['n_branches']))
print('§1 circuit k_max was :', _C.get('k_max_cfg'))

## §7 — Validation (faithfulness + class-relevant structure)

On the **circuit** tree: NMF init-stability per layer, causal-reconstruction fidelity (fc layers only), and the weight-term control (arbor-NMF vs activation-NMF separability). All via `src`; cached.

In [ ]:
from src import (compute_nmf_stability, summarize_validation, node_pos_arbor,
                 nodes_by_layer, weight_term_control, cached_result)
import numpy as _np
_C = ANALYSIS_CTX
def _run_validation():
    tc = _C['tree_circuit']
    nbl = nodes_by_layer(tc)
    stability = {}
    for li, nd in nbl.items():
        X = node_pos_arbor(nd, _C['layer_inputs'][li])
        k = int(nd.img_factors.shape[1])
        sim, _ = compute_nmf_stability(
            X if X.shape[0] <= 800 else X[_np.random.default_rng(0).choice(X.shape[0], 800, False)],
            k, n_seeds=_C.get('stab_seeds', 5), max_iter=300)
        stability[int(li)] = float(sim[~_np.eye(len(sim), dtype=bool)].mean())
    recon = summarize_validation(tc.nodes())        # None when no fc node was validated
    wtc = weight_term_control(tc, _C['labels_task'], _C['layer_inputs'])
    return {'stability': stability,
            'recon_overall': (recon['overall'] if recon else None),
            'weight_term': wtc}
val = cached_result(_C['tag'] + '_validation', _run_validation,
                    params=dict(n=len(_C['labels_task']), tag='circ'))
print('NMF stability per layer:', {k: round(v, 3) for k, v in val['stability'].items()})
if val['recon_overall']:
    print('causal recon preact_R2 (median):',
          round(val['recon_overall']['preact_r2']['median'], 3))
print('weight-term control: arbor NMF sil=%.3f  vs activation NMF sil=%.3f'
      % (val['weight_term']['arbor_nmf']['sil'], val['weight_term']['activation_nmf']['sil']))

## §8 — Causal pruning

Prunes each class circuit's weights in BFT-importance order on the **circuit** tree and measures target vs bystander accuracy (`src.pruning`, wrapping `ablation_sweep`). One seed here; add checkpoints for the full seed×class grid on the cluster. Cached.

In [ ]:
from src import run_pruning, cached_result
import numpy as _np
_C = ANALYSIS_CTX
if _C.get('skip_pruning'):
    print('pruning skipped for this model:', _C.get('skip_pruning'))
    prune = None
else:
    _reps = [{'seed': 0, 'model': _C['model'], 'tree': _C['tree_circuit'],
              'layer_names': _C.get('layer_names'), 'targets': _C['labels_task']}]
    _targets = (list(range(_C['n_classes'])) if _C['n_classes'] <= 10
                else list(range(_C['n_classes']))[:10])
    prune = cached_result(
        _C['tag'] + '_pruning',
        lambda: run_pruning(_reps, _C['eval_loader'], _targets, n_classes=_C['n_classes'],
                            fractions=_C.get('prune_fractions', (0.02, 0.05, 0.1, 0.2)),
                            label_transform=_C.get('label_transform'), device=_C.get('device'),
                            n_random_repeats=_C.get('n_random', 5), verbose=1)['aggregate'],
        params=dict(n=len(_targets), tag='prune'))
    for m in ('bft_top', 'bft_bottom', 'random'):
        if m in prune['methods']:
            print('%-11s target drop@0.2 (mean over classes): %+.3f'
                  % (m, _np.mean(prune['drops'][m]['target'])))

## §9 — Fingerprint separability (C1.8)

On the **fingerprint** tree: is the factor fingerprint more class-separable than the network's own activations, and where in the tree does that live? `src.separability` gives silhouette + kNN for the whole tree, its upper/lower slices, and the penultimate / full-activation baselines (native and dim-matched). Cached.

In [ ]:
from src import separability_evaluate, cached_result
_C = ANALYSIS_CTX
sep = cached_result(
    _C['tag'] + '_separability',
    lambda: separability_evaluate(_C['tree_fp'], _C['labels_fine'], _C['layer_inputs']),
    params=dict(n=len(_C['labels_fine']), tag='fp'))
_n = sep['native']
print('native silhouette:  fp_full=%.3f  output_only=%.3f  top_half=%.3f  spine=%.3f'
      % (_n['fp_full']['sil'], _n.get('fp_output_only', {}).get('sil', float('nan')),
         _n.get('fp_top_half', {}).get('sil', float('nan')), _n.get('fp_spine', {}).get('sil', float('nan'))))
print('activation baselines: penult=%.3f  full=%.3f'
      % (_n['act_penult']['sil'], _n['act_full']['sil']))
_p = sep['paired'].get('fp_full__vs__act_penult')
if _p:
    print('dim-matched @%d: fp(pca)=%.3f vs penult(pca)=%.3f'
          % (_p['match_dim'], _p['A_pca']['sil'], _p['B_pca']['sil']))